In [0]:
%pip install -U databricks-sdk==0.62.0 mlflow==3.2.0 databricks-agents==1.3.0 databricks-langchain==0.6.0 langchain==0.3.27 langchain_core==0.3.74 langgraph==0.6.4 langgraph-prebuilt==0.6.4 backoff
dbutils.library.restartPython()

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
from agent_mlflow3 import AGENT

input = {"input": [{"role": "user", "content": "What is 6*7 in Python?"}]}

result = AGENT.predict(input)
print(result.model_dump(exclude_none=True))

In [0]:
for chunk in AGENT.predict_stream(input):
    print(chunk.model_dump(exclude_none=True))

In [0]:
# Determine Databricks resources to specify for automatic auth passthrough at deployment time
from agent_mlflow3 import UC_TOOL_NAMES, VECTOR_SEARCH_TOOLS
import mlflow
from mlflow.models.resources import DatabricksFunction
from pkg_resources import get_distribution

mlflow.set_registry_uri("databricks-uc")

# TODO: define the catalog, schema, and model name for your UC model
catalog = "yen_training"
schema = "agents"
model_name = "langgraph-responses-agent"
UC_MODEL_NAME = f"{catalog}.{schema}.{model_name}"

resources = []
for tool in VECTOR_SEARCH_TOOLS:
    resources.extend(tool.resources)
for tool_name in UC_TOOL_NAMES:
    resources.append(DatabricksFunction(function_name=tool_name))

In [0]:
with mlflow.start_run():
    logged_agent_info = mlflow.pyfunc.log_model(
        name="agent",
        python_model="agent_mlflow3.py",
        registered_model_name=UC_MODEL_NAME,
        pip_requirements=[
            "databricks-langchain",
            f"langgraph=={get_distribution('langgraph').version}",
            f"backoff=={get_distribution('backoff').version}",
            f"databricks-connect=={get_distribution('databricks-connect').version}",
        ],
        resources=resources,
    )

In [0]:
from databricks import agents

agents.deploy(
    model_name=UC_MODEL_NAME,
    model_version=int(logged_agent_info.registered_model_version),
    scale_to_zero=True,
    tags={"endpointSource": "docs"}
)

In [0]:
model_uri = f"runs:/{logged_agent_info.run_id}/agent"
#model_uri = 'runs:/5a689e7116954b599aedcd5293717fdb/agent'
model_uri

In [0]:
loaded_model = mlflow.pyfunc.load_model(model_uri)

In [0]:
response = loaded_model.predict(input)
response